In [0]:
-- =====================================================================================
-- Notebook: 00_admin/01_volumes_setup.sql
-- Project : nba-datalakehouse
-- Purpose : Create Unity Catalog Volumes used as governed file storage for this project.
--
-- WHY VOLUMES?
-- -----------
-- In your Free Edition workspace:
--   - Hive Metastore is disabled (UC-only)
--   - Public DBFS root (e.g., dbfs:/FileStore) is disabled
--   - Direct local filesystem paths (file:/Workspace/...) are forbidden by policy
--
-- Unity Catalog Volumes solve this by giving you a secure, supported storage path:
--   dbfs:/Volumes/<catalog>/<schema>/<volume>/
--
-- We use volumes for:
--   1) landing  : raw JSONL files copied/uploaded into Databricks (Bronze files)
--   2) checkpoints (optional, v2): structured streaming state
--   3) exports  (optional): curated outputs for sharing
--
-- ORGANIZATION RECOMMENDATION
-- ---------------------------
-- Keep data in volumes, and keep notebooks in Workspace folders:
--   Workspace (code):  /Workspace/Users/.../nba-datalakehouse/notebooks/...
--   Volumes  (data):  dbfs:/Volumes/workspace/nba_files/landing/...
--
-- After running this notebook, you should be able to browse volumes via Catalog Explorer:
--   Catalog: workspace
--   Schema : nba_files
--   Volumes: landing, (optional) checkpoints, (optional) exports
--
-- NOTE
-- ----
-- You already successfully used a volume earlier, so this notebook makes that setup
-- explicit and reproducible.
-- =====================================================================================


-- =====================================================================================
-- 1) Ensure we are in the correct UC catalog + schema
-- =====================================================================================
USE CATALOG workspace;
CREATE SCHEMA IF NOT EXISTS nba_files
COMMENT 'Storage schema for UC Volumes (landing/checkpoints/exports) for NBA project.';
USE SCHEMA nba_files;


-- =====================================================================================
-- 2) Create the landing volume (required)
--    This is where raw JSONL files should live long-term inside Databricks.
--    Example final path:
--      dbfs:/Volumes/workspace/nba_files/landing/the_odds_api/events/dt=YYYY-MM-DD/part-*.jsonl
-- =====================================================================================
CREATE VOLUME IF NOT EXISTS landing
COMMENT 'Raw landing storage for NBA project (JSONL files partitioned by dt).';


-- =====================================================================================
-- 3) Optional volumes (recommended for future growth)
--    checkpoints:
--      - If you later add streaming ingestion (Auto Loader / Structured Streaming),
--        Spark needs a checkpoint location to store state.
--    exports:
--      - For exporting curated datasets (CSV/Parquet) to share outside Databricks.
-- =====================================================================================
CREATE VOLUME IF NOT EXISTS checkpoints
COMMENT 'Checkpoint storage for streaming jobs (future v2).';

CREATE VOLUME IF NOT EXISTS exports
COMMENT 'Export storage for curated outputs (optional).';


-- =====================================================================================
-- 4) Verify volumes exist
-- =====================================================================================
SHOW VOLUMES IN workspace.nba_files;


-- =====================================================================================
-- 5) Helpful reference (comment-only)
--
-- Once created, these are the canonical paths:
--   Landing:
--     dbfs:/Volumes/workspace/nba_files/landing/
--   Checkpoints:
--     dbfs:/Volumes/workspace/nba_files/checkpoints/
--   Exports:
--     dbfs:/Volumes/workspace/nba_files/exports/
--
-- Next notebook (recommended):
--   01_bronze/01_copy_landing_to_volume.py
--     - copies your already-unzipped Workspace files landing into the UC volume
-- OR (if already copied):
--   01_bronze/02_files_to_bronze_delta.py
--     - reads JSONL from the landing volume and writes Bronze Delta tables
-- =====================================================================================
